In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
pd.set_option('display.max_columns', None)


In [ ]:
# UNLP ror
institution_ror = catalog.load('params:openalex_fetch_options.institution_ror')

## Dimensión institución

In [ ]:
sat_openalex_institution = catalog.load('stg_openalex/sat_openalex_institution')
hub_openalex_institution = catalog.load('stg_openalex/hub_openalex_institution')
hub_openalex_ror = catalog.load('stg_openalex/hub_openalex_ror')
link_openalex_institution_ror = catalog.load('stg_openalex/link_openalex_institution_ror')

dim_institution_openalex = pd.merge(
    hub_openalex_institution[['institution_hk','institution_id']],
    sat_openalex_institution[['institution_hk','country_code','display_name']],
    on="institution_hk"
)

dim_institution_openalex = pd.merge(
    dim_institution_openalex,
    link_openalex_institution_ror[['institution_hk','ror_hk']],
    on="institution_hk"
)

dim_institution_openalex = pd.merge(
    dim_institution_openalex,
    hub_openalex_ror[['ror_hk','ror']],
    on="ror_hk"
)

filter_ror = dim_institution_openalex['ror'] == institution_ror

dim_institution_openalex[filter_ror][['institution_id','country_code','display_name','ror']]

## fact affiliation

In [ ]:
link_openalex_author_institution = catalog.load('stg_openalex/link_openalex_author_institution')
hub_openalex_author = catalog.load('stg_openalex/hub_openalex_author')
sat_openalex_affiliation = catalog.load('stg_openalex/sat_openalex_affiliation')

fact_affiliation_openalex = pd.merge(
    link_openalex_author_institution[['author_institution_hk','author_hk','institution_hk']],
    hub_openalex_author[['author_hk','author_id']],
    on='author_hk'
)

fact_affiliation_openalex = pd.merge(
    fact_affiliation_openalex,
    sat_openalex_affiliation[['author_institution_hk','years']],
    on="author_institution_hk",
)

fact_affiliation_openalex = pd.merge(
    fact_affiliation_openalex,
    hub_openalex_institution,
    on="institution_hk"
)

fact_affiliation_openalex = fact_affiliation_openalex[['author_id','institution_id','years','institution_hk']]
fact_affiliation_openalex

In [ ]:
fact_affiliation_openalex.groupby(['author_id','institution_id'])['years'].agg(list).reset_index()

Instituciones que comparten filiación con autores institucionales 

In [ ]:
pd.merge(
    dim_institution_openalex[['institution_hk','institution_id','display_name']],
    fact_affiliation_openalex[['institution_hk','author_id']],
    on="institution_hk"
).groupby(['institution_id','display_name']).count().sort_values(by="institution_hk", ascending=False)

# fact publication

In [ ]:
sat_openalex_work = catalog.load('stg_openalex/sat_openalex_work')
hub_openalex_work = catalog.load('stg_openalex/hub_openalex_work')

fact_publication_openalex = pd.merge(
    hub_openalex_work,
    sat_openalex_work
    ).drop(columns=['load_datetime','source', 'hashdiff'])

fact_publication_openalex[['title','publication_year','cited_by_count','oa_status']].sort_values(by='cited_by_count', ascending=False)

In [ ]:

# Contar publicaciones por año
publications_per_year = fact_publication_openalex['publication_year'].value_counts().sort_index()

# Graficar
plt.figure(figsize=(10, 5))
publications_per_year.plot(kind='bar', color='skyblue', edgecolor='black')
plt.xlabel('Año de publicación')
plt.ylabel('Cantidad de publicaciones')
plt.title('Publicaciones por año')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Mostrar gráfico
plt.show()